In [27]:
import json
import pandas as pd
from tqdm import tqdm
import re
import numpy as np
import matplotlib.pyplot as plt
import math

## Load Processed FDA file

In [30]:
df_orig = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full.csv")
df_orig.shape

(26306, 22)

In [31]:
df_orig.columns

Index(['application_number', 'sponsor_name', 'submission_type',
       'submission_status_date', 'submission_class_code_description', 'year',
       'brand_name', 'dosage_form', 'route', 'marketing_status',
       'active_ingredients', 'openfda_brand_name', 'openfda_generic_name',
       'openfda_route', 'openfda_substance_name', 'label_brand_name',
       'label_generic_name', 'label_manufacturer_name', 'label_substance_name',
       'indications_first_sent', 'indications_and_usage', 'clinical_studies'],
      dtype='object')

In [32]:
df_orig.head()

,application_number,sponsor_name,submission_type,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,marketing_status,...,openfda_generic_name,openfda_route,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
0,ANDA076194,WATSON LABS,ORIG,20020701,NaN,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,Prescription,...,LISINOPRIL AND HYDROCHLOROTHIAZIDE,ORAL,"HYDROCHLOROTHIAZIDE, LISINOPRIL",Lisinopril and Hydrochlorothiazide,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"Actavis Pharma, Inc.","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[]
1,ANDA076206,ROCKWELL MEDCL,ORIG,20030917,NaN,2003,CALCITRIOL,INJECTABLE,INJECTION,Discontinued,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ANDA076212,APOTEX,ORIG,20040616,NaN,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,Discontinued,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ANDA076215,FOUGERA PHARMS,ORIG,20031209,NaN,2003,BETAMETHASONE DIPROPIONATE,"CREAM, AUGMENTED",TOPICAL,Prescription,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ANDA076224,PHARMOBEDIENT,ORIG,20030509,NaN,2003,FLUTAMIDE,CAPSULE,ORAL,Discontinued,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_orig = df_orig[
    ~(df_orig["brand_name"].isna() | (df_orig["brand_name"].astype(str).str.strip() == "") |
      df_orig["active_ingredients"].isna() | (df_orig["active_ingredients"].astype(str).str.strip() == ""))
]
df_orig.shape

In [ ]:
# Step 1: Make sure the column is a proper list
df_orig["active_ingredients_split"] = (
    df_orig["active_ingredients"]
    .astype(str)  # ensure string
    .str.split(r",\s*")  # split on comma + optional spaces
)

# Step 2: Explode into separate rows
df_orig = df_orig.explode("active_ingredients_split", ignore_index=True)
df_orig.shape

In [ ]:
df_orig_info = df_orig[['sponsor_name','year','application_number', 'brand_name', 'active_ingredients_split', 'clinical_studies', 'indications_first_sent', 'indications_and_usage']]
df_orig_info.head()

## Load DS drugs

In [7]:
terms_path = "out/unique_drug_terms_292514.csv"
terms = pd.read_csv(terms_path)
terms_for_details = terms[terms['n_articles']>1]
terms_for_details = terms_for_details[terms_for_details["merged_umls_label"].str.len() > 2]

terms_for_details

,merged_umls_label,n_articles
0,Dexamethasone,5806
1,Acetylcysteine,4559
2,NG-Nitroarginine Methyl Ester,4306
3,Sirolimus,4107
4,Doxorubicin,4021
...,...,...
75386,abu,2
75387,βgh,2
75388,ASP7663,2
75389,5-hydroxytryptamine (5-ht) (2a/c) receptor ant...,2


## Merge drugs from dataset with FDA details

In [15]:
# 1) Build normalized join keys (upper + strip)
df_counts = terms_for_details.copy()
df_exploded = df_orig_info.copy()

df_counts["__key"] = df_counts["merged_umls_label"].astype(str).str.upper().str.strip()
df_exploded["__key"] = df_exploded["active_ingredients_split"].astype(str).str.upper().str.strip()

# 2) Left-merge ON the ingredient (keep all rows from df_exploded)
df_merged = df_counts.merge(
    df_exploded,
    on="__key",
    how="left"
)

# 3) Clean up
df_merged = df_merged.drop(columns="__key")
df_merged

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,clinical_studies,indications_first_sent,indications_and_usage
0,Dexamethasone,5806,SUN PHARM INDUSTRIES,1974.0,ANDA084013,DEXAMETHASONE,DEXAMETHASONE,NaN,NaN,NaN
1,Dexamethasone,5806,WHITEWORTH TOWN PLSN,1975.0,ANDA084327,DEXAMETHASONE,DEXAMETHASONE,NaN,NaN,NaN
2,Dexamethasone,5806,WATSON LABS,1977.0,ANDA085455,DEXAMETHASONE,DEXAMETHASONE,NaN,NaN,NaN
3,Dexamethasone,5806,HIKMA,1983.0,ANDA088248,DEXAMETHASONE,DEXAMETHASONE,[],INDICATIONS AND USAGE Allergic States Control ...,INDICATIONS AND USAGE Allergic States Control ...
4,Dexamethasone,5806,EYEPOINT PHARMS,2018.0,NDA208912,DEXYCU KIT,DEXAMETHASONE,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
91755,abu,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91756,βgh,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91757,ASP7663,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91758,5-hydroxytryptamine (5-ht) (2a/c) receptor ant...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df_ds_drugs_with_FDA_info = df_merged[df_merged["application_number"].notna()]
df_ds_drugs_with_FDA_info

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,clinical_studies,indications_first_sent,indications_and_usage
0,Dexamethasone,5806,SUN PHARM INDUSTRIES,1974.0,ANDA084013,DEXAMETHASONE,DEXAMETHASONE,NaN,NaN,NaN
1,Dexamethasone,5806,WHITEWORTH TOWN PLSN,1975.0,ANDA084327,DEXAMETHASONE,DEXAMETHASONE,NaN,NaN,NaN
2,Dexamethasone,5806,WATSON LABS,1977.0,ANDA085455,DEXAMETHASONE,DEXAMETHASONE,NaN,NaN,NaN
3,Dexamethasone,5806,HIKMA,1983.0,ANDA088248,DEXAMETHASONE,DEXAMETHASONE,[],INDICATIONS AND USAGE Allergic States Control ...,INDICATIONS AND USAGE Allergic States Control ...
4,Dexamethasone,5806,EYEPOINT PHARMS,2018.0,NDA208912,DEXYCU KIT,DEXAMETHASONE,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
88816,Relebactam,2,MSD MERCK CO,2019.0,NDA212819,RECARBRIO,RELEBACTAM,"['NCT02493764', 'NCT01505634', 'NCT01506271']",1 INDICATIONS AND USAGE RECARBRIO is a combina...,1 INDICATIONS AND USAGE RECARBRIO is a combina...
89090,Isocarboxazid,2,LIFSA DRUGS,1959.0,NDA011961,MARPLAN,ISOCARBOXAZID,NaN,NaN,NaN
90792,selpercatinib,2,ELI LILLY AND CO,2024.0,NDA218160,RETEVMO,SELPERCATINIB,"['NCT03157128', 'NCT04194944', 'NCT03157128', ...",1 INDICATIONS AND USAGE RETEVMO ® is a kinase ...,1 INDICATIONS AND USAGE RETEVMO ® is a kinase ...
90793,selpercatinib,2,ELI LILLY AND CO,2020.0,NDA213246,RETEVMO,SELPERCATINIB,"['NCT03157128', 'NCT04194944', 'NCT03157128', ...",1 INDICATIONS AND USAGE RETEVMO ® is a kinase ...,1 INDICATIONS AND USAGE RETEVMO ® is a kinase ...


In [18]:
df_with_indications = df_ds_drugs_with_FDA_info[df_ds_drugs_with_FDA_info["indications_first_sent"].notna()]
df_with_indications['unique_id'] = (
    df_with_indications['merged_umls_label']
    .str.replace(" ", "_")
    .str.lower()
    +"_" + df_with_indications['application_number'].astype(str)
)


/tmp/ipykernel_678012/2705641560.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_indications['unique_id'] = (


In [19]:
df_with_indications.head()

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,clinical_studies,indications_first_sent,indications_and_usage,unique_id
3,Dexamethasone,5806,HIKMA,1983.0,ANDA088248,DEXAMETHASONE,DEXAMETHASONE,[],INDICATIONS AND USAGE Allergic States Control ...,INDICATIONS AND USAGE Allergic States Control ...,dexamethasone_ANDA088248
5,Dexamethasone,5806,SANDOZ,1988.0,NDA050592,TOBRADEX,DEXAMETHASONE,[],INDICATIONS AND USAGE Tobramycin and dexametha...,INDICATIONS AND USAGE Tobramycin and dexametha...,dexamethasone_NDA050592
6,Dexamethasone,5806,PADAGIS US,1989.0,ANDA062938,NEOMYCIN AND POLYMYXIN B SULFATES AND DEXAMETH...,DEXAMETHASONE,[],INDICATIONS AND USAGE: For steroid-responsive ...,INDICATIONS AND USAGE: For steroid-responsive ...,dexamethasone_ANDA062938
7,Dexamethasone,5806,BAUSCH AND LOMB,1995.0,ANDA064135,DEXASPORIN,DEXAMETHASONE,[],INDICATIONS AND USAGE For steroid-responsive i...,INDICATIONS AND USAGE For steroid-responsive i...,dexamethasone_ANDA064135
8,Dexamethasone,5806,PRASCO,1971.0,ANDA080399,DEXAMETHASONE,DEXAMETHASONE,[],INDICATIONS AND USAGE Allergic States Control ...,INDICATIONS AND USAGE Allergic States Control ...,dexamethasone_ANDA080399


In [25]:
df_with_indications.shape

(8779, 11)

### save for LLM processing

In [23]:
df_with_indications.to_csv("out/drugs_with_indications.csv", index=False)
df_with_indications.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/drugs_with_indications.csv", index=False)


In [29]:
# Calculate chunk size
num_chunks = 3
chunk_size = math.ceil(len(df_with_indications) / num_chunks)

# Split and save each chunk
for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = start_idx + chunk_size
    chunk_df = df_with_indications.iloc[start_idx:end_idx]
    print(f"chunk size {chunk_df.shape}")
    chunk_df.to_csv(f"out/drugs_with_indications_chunk_{i+1}.csv", index=False)


chunk size (2927, 11)
chunk size (2927, 11)
chunk size (2925, 11)


In [24]:
df_ds_drugs_with_FDA_info.merged_umls_label.nunique()

1354

In [20]:
df_filtered = df_ds_drugs_with_FDA_info[
    (df_ds_drugs_with_FDA_info["active_ingredients_split"] == "DEXAMETHASONE") &
    (df_ds_drugs_with_FDA_info["application_number"].astype(str).str.startswith("NDA"))
]
df_filtered

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,clinical_studies,indications_first_sent,indications_and_usage
4,Dexamethasone,5806,EYEPOINT PHARMS,2018.0,NDA208912,DEXYCU KIT,DEXAMETHASONE,NaN,NaN,NaN
5,Dexamethasone,5806,SANDOZ,1988.0,NDA050592,TOBRADEX,DEXAMETHASONE,[],INDICATIONS AND USAGE Tobramycin and dexametha...,INDICATIONS AND USAGE Tobramycin and dexametha...
19,Dexamethasone,5806,ABBVIE,2009.0,NDA022315,OZURDEX,DEXAMETHASONE,[],1 INDICATIONS AND USAGE OZURDEX ® is a cortico...,1 INDICATIONS AND USAGE OZURDEX ® is a cortico...
20,Dexamethasone,5806,ASPEN GLOBAL INC,1960.0,NDA012674,HEXADROL,DEXAMETHASONE,NaN,NaN,NaN
21,Dexamethasone,5806,ASPEN GLOBAL INC,1962.0,NDA012675,HEXADROL,DEXAMETHASONE,NaN,NaN,NaN
22,Dexamethasone,5806,OCULAR THERAPEUTIX,2018.0,NDA208742,DEXTENZA,DEXAMETHASONE,"['NCT02034019', 'NCT02089113', 'NCT02736175', ...",1 INDICATIONS AND USAGE DEXTENZA ® is a cortic...,1 INDICATIONS AND USAGE DEXTENZA ® is a cortic...
23,Dexamethasone,5806,HARROW EYE,2009.0,NDA050818,TOBRADEX ST,DEXAMETHASONE,[],1 INDICATIONS AND USAGE TOBRADEX ST ophthalmic...,1 INDICATIONS AND USAGE TOBRADEX ST ophthalmic...
36,Dexamethasone,5806,HARROW EYE,1963.0,NDA050023,MAXITROL,DEXAMETHASONE,[],INDICATIONS AND USAGE For steroid-responsive i...,INDICATIONS AND USAGE For steroid-responsive i...
37,Dexamethasone,5806,NOVARTIS,1988.0,NDA050616,TOBRADEX,DEXAMETHASONE,[],INDICATIONS AND USAGE: TOBRADEX (tobramycin an...,INDICATIONS AND USAGE: TOBRADEX (tobramycin an...
40,Dexamethasone,5806,DEXCEL,2019.0,NDA211379,HEMADY,DEXAMETHASONE,[],1 INDICATIONS AND USAGE HEMADY is indicated in...,1 INDICATIONS AND USAGE HEMADY is indicated in...


In [21]:
df_ds_drugs_with_FDA_info[df_ds_drugs_with_FDA_info['active_ingredients_split']=="RITUXIMAB"].indications_first_sent.iloc[0]

"1 INDICATIONS AND USAGE RITUXAN is a CD20-directed cytolytic antibody indicated for the treatment of: Adult patients with Non-Hodgkin's Lymphoma (NHL) ( 1.1 )"

In [22]:
df_ds_drugs_with_FDA_info[df_ds_drugs_with_FDA_info['active_ingredients_split']=="RITUXIMAB"].indications_first_sent.iloc[1]

'1 INDICATIONS AND USAGE RITUXAN HYCELA is a combination of rituximab, a CD20-directed cytolytic antibody, and hyaluronidase human, an endoglycosidase, indicated for the treatment of adult patients with: Follicular Lymphoma (FL) ( 1.1 ) Relapsed or refractory, follicular lymphoma as a single agent Previously untreated follicular lymphoma in combination with first line chemotherapy and, in patients achieving a complete or partial response to rituximab in combination with chemotherapy, as single-agent maintenance therapy Non-progressing (including stable disease), follicular lymphoma as a single agent after first-line cyclophosphamide, vincristine, and prednisone (CVP) chemotherapy Diffuse Large B-cell Lymphoma (DLBCL) ( 1.2 ) Previously untreated diffuse large B-cell lymphoma in combination with cyclophosphamide, doxorubicin, vincristine, prednisone (CHOP) or other anthracycline-based chemotherapy regimens Chronic Lymphocytic Leukemia (CLL) ( 1.3 ) Previously untreated and previously tr